In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


In [ ]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
display(df.head())

In [ ]:
df.groupby("Demand_Response_Flag")['Site'].count()

In [ ]:
def preprocess_data(df_in):
    df = df_in.copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    df.drop(columns=['Timestamp_Local','Timestamp','Site','Demand_Response_Capacity_kW'], inplace=True)
    # Fix target variable (instead of -1 make it 2)
    df['Demand_Response_Flag'] = df['Demand_Response_Flag'].replace(-1, 2)
    return df

df = preprocess_data(df)

In [ ]:
display(df.head(), df.tail())

In [ ]:
# Assume the target column is 'Demand_Response_Flag' (3 classes: 0, 1, 2)
# If not, replace with the correct target column

# Prepare features and target
X = df.drop(columns=['Demand_Response_Flag']).values
y = df['Demand_Response_Flag'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Define neural network
class Net(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, num_classes)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

input_dim = X_train.shape[1]
num_classes = 3

# create a neural net model
model = Net(input_dim, num_classes)

# Calculate class weights to handle class imbalance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
# Update criterion to use class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# configure optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_tensor).float().mean().item()
    print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
joblib.dump(scaler, 'scaler.pkl')
# Save the trained model to a file
torch.save(model.state_dict(), 'trained_model.pth')

In [ ]:
# Load the trained model from file
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('trained_model.pth'))
model_loaded.eval()

In [ ]:
df.head()